In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from global_vars import chargement_df

In [ ]:
df = pd.read_csv("../data/raw/accidents_2019_2023.csv")

In [ ]:
# changement ordre de gravité pour avoir un ordre logique 
df['grav'] = df['grav'].replace({2: 42})
df['grav'] = df['grav'].replace({4: 2})
df['grav'] = df['grav'].replace({42: 4})

In [ ]:
# modification de lat et long pour avoir le bon style

df['lat'] = df['lat'].str.replace(',', '.')
df['long'] = df['long'].str.replace(',', '.')

df['lat'] = df['lat'].astype(float)
df['long'] = df['long'].astype(float)

In [ ]:
# création d'une variable age au moment de l'accident
df['age'] = df['an']-df['an_nais']
df.drop(['an_nais'],axis=1,inplace = True)

# remplace les valeurs abérantes > 105 ans

df['age'] = df['age'].replace({106: 6})
df['age'] = df['age'].replace({108: 8})
df['age'] = df['age'].replace({109: 9})
df['age'] = df['age'].replace({110: 10})
df['age'] = df['age'].replace({118: 18})
df['age'] = df['age'].replace({119: 19})
df['age'] = df['age'].replace({120: 20})

# fonction pour ramplacer des valeurs manquantes dans age par une valeur aléatoire parmis 
# les valeurs observées en respectant la distribution observée

def replace_nan_with_random(df, column_name):
    observed_values = df[column_name].dropna()
    random_values = np.random.choice(observed_values, size=df[column_name].isna().sum())
    df.loc[df[column_name].isna(), column_name] = random_values
    return df

# Remplacer les NaN dans 'age'
df = replace_nan_with_random(df, 'age')



In [ ]:
# suppression des valeurs manquantes pour la variable cible

df = df[accidents_copy['grav'] != -1]

In [ ]:
# Définition de la fonction pour homogénéiser le format de l'heure
def homogenize_hour_format(row):
    # Convertir l'heure en chaîne de caractères
    hour_str = str(row['hrmn'])
    
    # Si l'année est comprise entre 2005 et 2018, ajuster le format de l'heure
    if row['an'] < 2019:
        # Extraire les deux derniers chiffres pour les minutes
        minutes = hour_str[-2:].zfill(2)
        
        # extrait les autre pour les heures
        hour = hour_str[:-2].zfill(2)
        
        #concatene avec ':' pour obtenir le format 'HH:MM'
        return f'{hour}:{minutes}'
    
    # Si l'année est 2019 ou plus, le format est déjà 'HH:MM'
    return hour_str

# Appliquer la fonction à la colonne 'hrmn'
df['hrmn'] = df.apply(homogenize_hour_format, axis=1)

In [ ]:
# remplacement des valeurs manquante de sexe suivant leur proportion dans la catégorie
# grav = 1 (toutes les valeurs manquantes sont dans cette catégorie)

proportion = [1] * 71 + [2] * 29

n_manquants = (df['sexe'] == -1).sum()
valeurs_remplacement = np.random.choice(proportion, size=n_manquants, replace=True)
df.loc[df['sexe'] == -1, 'sexe'] = valeurs_remplacement

In [ ]:
# remplacement des valeurs abérrantes vma
valeurs_50 = [500, 55, 520, 501, 502, 5]
valeurs_90 = [900, 901, 9]
valeurs_80 = [800, 180, 8]
valeurs_70 = [7, 75, 700, 770]
valeurs_60 = [560, 600]
valeurs_30 = [3, 300, 31, 35]
valeurs_40 = [140, 4, 42]

df['vma'] = df['vma'].replace(valeurs_50, 50)
df['vma'] = df['vma'].replace(valeurs_70, 70)
df['vma'] = df['vma'].replace(valeurs_80, 80)
df['vma'] = df['vma'].replace(valeurs_90, 90)
df['vma'] = df['vma'].replace(valeurs_60, 60)
df['vma'] = df['vma'].replace(valeurs_30, 30)
df['vma'] = df['vma'].replace(valeurs_40, 40)

# remplacement des -1 en fonction de la catégorie de route

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 1), 'vma'] = 130

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 2), 'vma'] = 80

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 6), 'vma'] = 30

df['vma'] = df['vma'].replace({-1: 50})

df.loc[df['vma'] < 30, 'vma'] = 30


# regroupement des catégories rares avec une catégorie proche

df['vma'] = df['vma'].replace({40: 50})
df['vma'] = df['vma'].replace({45: 50})
df['vma'] = df['vma'].replace({60: 70})
df['vma'] = df['vma'].replace({65: 70})
df['vma'] = df['vma'].replace({100: 110})
df['vma'] = df['vma'].replace({120: 130})
                                                      
# les vitesses à 130 hors autoroute ramenées à 30

df.loc[(df['vma'] == 130) &
                   (df['catr'] != 1), 'vma'] = 30

#les vitesses >70 sur voies cummunales ramenées à 70
df.loc[(df['vma'] > 70) &
                   (df['catr'] == 4), 'vma'] = 70



In [ ]:
# remplacement des valeurs manquantes de lum en fonction des heures de la journée

# Remplacer les -1 par 1 entre 9h et 17h
df.loc[(df['lum'] == -1) &
                   (df['hrmn'] > '09:00') &
                   (df['hrmn'] < '17:00'), 'lum'] = 1

# Remplacer les -1 par 3 avant 7h ou après 20h
df.loc[(df['lum'] == -1) &
                   ((df['hrmn'] < '07:00') |
                    (df['hrmn'] > '20:00')), 'lum'] = 3

# Remplacer les -1 restants par 2
df['lum'] = df['lum'].replace({-1: 2})

In [ ]:
# remplacement des NAs par le mode de la colonne pour toutes les variables concernées
imputer = SimpleImputer(strategy='most_frequent')
cols_to_impute = [''] # A remplir avec les colonnes concernées
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

In [ ]:
# Suppression des colonnes avec plus de 90% de valeurs manquantes
seuil = 0.9  # 90%
nb_lignes = len(df)
colonnes_a_supprimer = [col for col in df.columns if df[col].isna().sum() / nb_lignes > seuil]

print("Colonnes supprimées :", colonnes_a_supprimer)
df.drop(columns=colonnes_a_supprimer, inplace=True)

In [ ]:
# Création d'une colonne 'date' à partir de jour/mois/an
df['date'] = pd.to_datetime(dict(year=df['an'], month=df['mois'], day=df['jour']), errors='coerce')

# Création du jour de la semaine
df['jour_semaine'] = df['date'].dt.dayofweek  

# Extraction de l'heure depuis la colonne 'hrmn' s
def extraire_heure(chaine):
    try:
        return int(str(chaine).split(':')[0])
    except:
        return np.nan

df['heure'] = df['hrmn'].apply(extraire_heure)

In [ ]:
# Liste des colonnes catégorielles à encoder
colonnes_label = ['lum', 'agg', 'catr', 'atm', 'col', 'plan', 'surf', 'infra', 'situ']

# Encodage LabelEncoder
for col in colonnes_label:
    df[col] = df[col].astype('category').cat.codes

In [ ]:
# Convertir toutes les colonnes catégorielles 
cat_cols = df_clean.select_dtypes(include='object').columns


encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_clean[cat_cols] = encoder.fit_transform(df_clean[cat_cols])


In [ ]:
# Suppression des colonnes inutiles
colonnes_inutiles = ['Num_Acc', 'id_vehicule', 'voie', 'v1', 'v2', 'pr', 'pr1', 'adr', 'hrmn', 'an_nais', 'date']

df.drop(columns=[col for col in colonnes_inutiles if col in df.columns], inplace=True)